# Exploring the Role of the Complement Cascade in CKD and AKI
> Side-mission is to create a Complement Cascade Atlas in the Human Kidney

## Download data from CZ CELLxGENE

Download data from CZ CELLxGENE using [TODO] and aquire citation for data.

In [ ]:
!mkdir /Users/aumchampaneri/VSCode/hs-ckd-aki/data/

In [ ]:
import cellxgene_census
census = cellxgene_census.open_soma(census_version="latest")
census["census_info"]["summary"].read().concat().to_pandas()
datasets = census["census_info"]["datasets"].read().concat().to_pandas()

In [ ]:
dataset_id = "7ff0197b-d175-49bf-b4fa-150fe0995d93"
datasets[datasets["dataset_id"] == dataset_id].iloc[0]

In [ ]:
# BROKEN - MANUALLY DOWNLOADED FROM CELLXGENE CENSUS UI
cellxgene_census.download_source_h5ad(
    "f337b525-c8f7-4c96-8cfe-f258a9f5ca48", 
    to_path="/Users/aumchampaneri/VSCode/hs-ckd-aki/data/f337b525-c8f7-4c96-8cfe-f258a9f5ca48.h5ad", 
    census_version="latest",
    progress_bar=True
)
census.close()

## scVI Processing



In [1]:
import scanpy as sc
import scvi
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

In [2]:
!mkdir /Users/aumchampaneri/VSCode/hs-ckd-aki/outputs/scVI

mkdir: /Users/aumchampaneri/VSCode/hs-ckd-aki/outputs/scVI: File exists


In [3]:
adata = sc.read_h5ad("/Users/aumchampaneri/VSCode/hs-ckd-aki/data/f337b525-c8f7-4c96-8cfe-f258a9f5ca48.h5ad")
adata

AnnData object with n_obs × n_vars = 1388643 × 35455
    obs: 'nCount_RNA', 'nFeature_RNA', 'library', 'percent.er', 'percent.mt', 'experiment_id', 'suspension_type', 'assay_ontology_term_id', 'donor_id', 'specimen', 'Conditions.Subtype', 'disease_ontology_term_id', 'diabetes_history', 'hypertension', 'sex_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'region', 'percent.cortex', 'percent.medulla', 'tissue_type', 'tissue_ontology_term_id', 'Clusters', 'SubclassLevel3', 'SubclassLevel2', 'SubclassLevel1', 'CellStateLevel2', 'CellStateLevel1', 'Class', 'cell_type_ontology_term_id', 'ConditionCategory', 'TissueCollection', 'Race', 'AdjudicationCategory', 'Baseline eGFR (ml/min/1.73m2) (Binned)', 'SubclassLevel3_FullName', 'Age', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'fe

### Inspection

In [4]:
adata.obs[
    [
        "nFeature_RNA",
        "nCount_RNA",
        "percent.mt"
    ]
].describe()

,nFeature_RNA,nCount_RNA,percent.mt
count,1.388643e+06,1.388643e+06,1.388643e+06
mean,2.200280e+03,4.594136e+03,2.771208e+00
std,1.169948e+03,3.922160e+03,4.998688e+00
min,3.880000e+02,4.200000e+02,0.000000e+00
25%,1.347000e+03,2.029000e+03,5.390254e-01
50%,2.017000e+03,3.509000e+03,1.352657e+00
75%,2.820000e+03,5.806000e+03,2.950192e+00
max,7.487000e+03,5.256300e+04,9.477244e+01


In [5]:
adata.raw
adata.raw.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 3038706327 stored elements and shape (1388643, 35455)>

In [6]:
adata.raw.X.shape
adata.raw.X[:5, :5]

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 0 stored elements and shape (5, 5)>

In [7]:
adata.X = adata.raw.X.copy()


### HVG: Selction-Subset-Store

In [8]:
scvi.data.poisson_gene_selection(adata)

/Users/aumchampaneri/.pyenv/versions/bioenv/lib/python3.10/site-packages/scvi/data/_preprocessing.py:105: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  _, _, device = parse_device_args(


Sampling from binomial...:   0%|          | 0/10000 [00:00<?, ?it/s]

In [9]:
adata = adata[:, adata.var["highly_variable"]].copy()

In [10]:
adata.layers["counts"] = adata.X.copy()

if not sp.issparse(adata.layers["counts"]):
    adata.layers["counts"] = sp.csr_matrix(adata.layers["counts"])

In [11]:
adata.obs["batch"] = (
    adata.obs["donor_id"].astype(str)
    + "_"
    + adata.obs["library"].astype(str)
)

adata.obs["batch"] = adata.obs["batch"].astype("category")

### scVI: Setup-Initialize-Train

In [12]:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
    batch_key="batch"
)

In [ ]:
model = scvi.model.SCVI(
    adata,
    n_layers=2,
    n_hidden=256,
    n_latent=30,
    dropout_rate=0.1,
    gene_likelihood="nb" # We use Negative Binomial count likelihoods, following Boyeau et al., 2023.
)

model.train(
    max_epochs=300,
    batch_size=2048,
    early_stopping=True,
    early_stopping_patience=20,
    early_stopping_monitor="elbo_validation",
    accelerator="mps",
    plan_kwargs={"lr": 1e-3},
)

/Users/aumchampaneri/.pyenv/versions/bioenv/lib/python3.10/site-packages/scvi/train/_trainrunner.py:69: UserWarning: `accelerator` has been set to `mps`. Please note that not all PyTorch/Jax operations are supported with this backend. as a result, some models might be slower and less accurate than usuall. Please verify your analysis!Refer to https://github.com/pytorch/pytorch/issues/77764 for more details.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/aumchampaneri/.pyenv/versions/bioenv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.
/Users/aumchampaneri/.pyenv/versions/bioenv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_

Training:   0%|          | 0/300 [00:00<?, ?it/s]

In [ ]:
# Ensure convergence
train_test_results = model.history["elbo_train"]
train_test_results["elbo_validation"] = model.history["elbo_validation"]
train_test_results.iloc[10:].plot(logy=True)  # exclude first 10 epochs
plt.show()

### Extraction and Embedding

In [ ]:
adata.obsm["X_scVI"] = model.get_latent_representation()

In [ ]:
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=15)
sc.tl.umap(adata, min_dist=0.3)

### Export

In [ ]:
adata.write_h5ad("kidney_scvi_atlas_rawcounts.h5ad", compression="gzip")
model.save("scvi_kidney_model", overwrite=True)